# **Attach a Microsoft Foundry Agent** via Responses API (Azure OpenAI-compatible)

This notebook demonstrates how to **invoke a published Agent from Microsoft Foundry** using the **OpenAI‑compatible Responses API** exposed by Foundry “Applications”. Once an Agent is **published**, Foundry provides an **OpenAI protocol endpoint** under your project; you can call `POST /responses` exactly like the OpenAI Responses API, pointing your SDK to that **base URL**. [1](https://www.georgeollis.com/consuming-a-microsoft-foundry-agent-programmatically/)

**Key points:**
- Use the Foundry **Application OpenAI protocol endpoint** (pattern:  
  `https://{account}.services.ai.azure.com/api/projects/{project}/applications/{application}/protocols/openai`) as your SDK `base_url`, then call `POST /responses`. [1](https://www.georgeollis.com/consuming-a-microsoft-foundry-agent-programmatically/)
- **Microsoft Entra ID (OAuth) required**: Applications currently **do not** support API keys; ensure your caller has `applications/invoke/action` permission (e.g., **Azure AI User** role). [1](https://www.georgeollis.com/consuming-a-microsoft-foundry-agent-programmatically/)
- Application endpoints expose only **`POST /responses`** and force `store=false`, so **multi‑turn state is client‑managed** (you re‑send the relevant history each turn). [1](https://www.georgeollis.com/consuming-a-microsoft-foundry-agent-programmatically/)
- For Foundry model endpoints (non‑application), the **OpenAI v1 path** is `https://<resource>.services.ai.azure.com/openai/v1/` and supports the Responses surface with standard OpenAI clients. [2](https://learn.microsoft.com/en-us/agent-framework/user-guide/agents/agent-types/azure-ai-foundry-models-responses-agent)[3](https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/ai-foundry/openai/how-to/responses.md)

> **Why Responses API for agents?**  
> The Responses API in Azure AI Foundry is purpose‑built for agentic workflows (tool calling, multi‑turn, streaming, computer use), and is now GA. Published Agents can be invoked over this OpenAI‑compatible interface. [4](https://azurefeeds.com/2025/08/28/the-responses-api-in-azure-ai-foundry-is-now-generally-available/)[5](https://azure.microsoft.com/en-us/blog/announcing-the-responses-api-and-computer-using-agent-in-azure-ai-foundry/)


## Prerequisites & Environment Variables

- Python 3.10+  
- `pip install openai azure-identity python-dotenv`
- A **published Foundry Agent (Application)** with OpenAI protocol enabled (the portal shows the endpoint after publishing). [6](https://microsoft.github.io/copilot-camp/pages/custom-engine/agents-sdk/01-agent-in-foundry/)
- Azure RBAC: your identity must be able to **invoke** the Application (e.g., **Azure AI User** at project scope). [1](https://www.georgeollis.com/consuming-a-microsoft-foundry-agent-programmatically/)

Set the following environment variables:

- `FOUNDRY_APP_OPENAI_BASE_URL` → your Application’s OpenAI protocol base (e.g., `https://<account>.services.ai.azure.com/api/projects/<project>/applications/<application>/protocols/openai`)
- Optional if you use `DefaultAzureCredential`:  
  `AZURE_TENANT_ID`, `AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET`  
  or sign in with `az login` to use **Azure CLI credential** [2](https://learn.microsoft.com/en-us/agent-framework/user-guide/agents/agent-types/azure-ai-foundry-models-responses-agent)


# Constants and Libraries

In [1]:
import os
from IPython.display import Markdown, display
from dotenv import load_dotenv # package python-dotenv
from openai import AzureOpenAI # package openai
from azure.identity import DefaultAzureCredential

if not load_dotenv("./../../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")

else:
    print("Environment variables have been loaded ;-)")

application_endpoint = os.environ["AIF_BAS_APPLICATION_ENDPOINT"]
first_question = "What's the expected temperature in Paris tomorrow?"
follow_up_question = "What was my previous question?"
myAgent = "foundryagent-V2-with-bing"

Environment variables have been loaded ;-)


## Acquire bearer token for Foundry OpenAI protocol

In [2]:
def acquire_bearer_token(scope: str = "https://ai.azure.com/.default") -> str:
    """
    Acquire a Microsoft Entra token for Foundry endpoints.
    Prefers DefaultAzureCredential; falls back to AzureCliCredential.
    """
    try:
        cred = DefaultAzureCredential(exclude_powershell_credential=True)
        token = cred.get_token(scope)
        return token.token
    except Exception:
        # Fallback to Azure CLI if DefaultAzureCredential was not configured
        print("Fallback to Azure CLI since DefaultAzureCredential was not configured")
        cred = AzureCliCredential()
        token = cred.get_token(scope)
        return token.token

bearer = acquire_bearer_token()
print(f"Bearer token: {bearer[:50]}...")

Bearer token: eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIsIng1dCI6IlBjWD...


## Build OpenAI client pointing at the Application OpenAI protocol endpoint

In [3]:
# Note: Applications require bearer token auth; API keys are not supported at this time.
client = AzureOpenAI(
    base_url=application_endpoint.rstrip("/") + "/",
    api_key="__unused__",  # placeholder; we will send OAuth header
    default_headers={"Authorization": f"Bearer {bearer}"},
    api_version = os.getenv("AZURE_OPENAI_FOR_PUBLISHED_AGENTS_API_VERSION"), # "2025-11-15-preview"
)

print("Client initialized for:", client.base_url)

Client initialized for: https://aif1bassvj36b.services.ai.azure.com/api/projects/aif1basswcprj01/applications/foundryagent-V2-with-bing/protocols/openai/


# Invoke simple single-turn request to the published Agent
**Important note**: applications force store=false, so you cannot rely on server-side state.

In [4]:
resp1 = client.responses.create(
    input=first_question,
)
print(resp1.output_text)

The expected temperature in Paris tomorrow will range from a low of around -2°C to a high of about 3°C, with mostly sunny skies and minimal cloud cover. It will be chilly and colder than the January average, so dressing warmly is recommended.


## Follow-up question
The history has to be built manually